# Homework 7 - BERT Question Answering

**任務**：中文抽取式問答（Extractive QA）
- 輸入：段落 + 問題
- 輸出：答案在段落中的起始/結束位置

**改進重點（相對於 Simple baseline）**：
- **Medium**：`doc_stride` 縮小（重疊 window）+ Linear LR decay with warmup
- **Strong**：換用 `hfl/chinese-roberta-wwm-ext` + Preprocessing 修正 None token
- **Boss**：Postprocessing 修正（限制預測在段落範圍、確保 end >= start）

## 1. 下載資料集

In [ ]:
# 下載資料集
!gdown --id '1znKmX08v9Fygp-dgwo7BKiLIf2qL1FH1' --output hw7_data.zip

# 若上面失敗，改用備用連結
# !gdown --id '1pOu3FdPdvzielUZyggeD7KDnVy9iW1uC' --output hw7_data.zip

!unzip -o hw7_data.zip

# 確認 GPU 型號（V100 > T4 > P4 > K80）
!nvidia-smi

## 2. 安裝套件

In [5]:
# Colab 已預裝新版 transformers，不需重新安裝
# 只需安裝 accelerate（支援 fp16 混合精度訓練）
!pip install -q accelerate

## 3. Import 套件

In [6]:
import json
import numpy as np
import random
import torch
from torch.optim import AdamW  # 新版 transformers 已移除 AdamW，改從 torch.optim import
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModelForQuestionAnswering,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"使用裝置：{device}")

# 固定亂數種子，確保結果可重現
def same_seeds(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

same_seeds(0)

使用裝置：cuda


## 4. FP16 混合精度設定

In [7]:
# fp16（半精度）能大幅縮短訓練時間，建議開啟
# T4 GPU 上 fp16 訓練速度約為 fp32 的 2-3 倍
fp16_training = True

if fp16_training:
    from accelerate import Accelerator
    # 新版 accelerate 用 mixed_precision="fp16"，舊版用 fp16=True
    try:
        accelerator = Accelerator(mixed_precision="fp16")
    except TypeError:
        accelerator = Accelerator(fp16=True)
    device = accelerator.device
    print(f"FP16 啟用，裝置：{device}")

FP16 啟用，裝置：cuda


## 5. 載入預訓練模型

**Strong baseline 改動**：換用 `hfl/chinese-roberta-wwm-ext`
- 相比 `bert-base-chinese`，這個模型使用 Whole Word Masking（全詞遮蔽）預訓練
- 在中文 NLP 任務上普遍有顯著提升

In [24]:
# Strong: 使用更強的中文預訓練模型
# bert-base-chinese 公開分數約 0.446（Simple），換模型後能達到 Strong 以上
model_name = "hfl/chinese-roberta-wwm-ext"

model = AutoModelForQuestionAnswering.from_pretrained(model_name).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"模型載入完成：{model_name}")
# 警告訊息可以忽略（QA head 是隨機初始化的，因為是新任務的輸出層）

Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_http.py", line 761, in hf_raise_for_status
    response.raise_for_status()
  File "/usr/local/lib/python3.12/dist-packages/httpx/_models.py", line 829, in raise_for_status
    raise HTTPStatusError(message, request=request, response=self)
httpx.HTTPStatusError: Client error '403 Forbidden' for url 'https://huggingface.co/api/models/hfl/chinese-roberta-wwm-ext/discussions?p=0'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_convers

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: hfl/chinese-roberta-wwm-ext
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
qa_outputs.bias                            | MISSING    | 
qa_outputs.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expec

模型載入完成：hfl/chinese-roberta-wwm-ext


## 6. 讀取資料

資料格式：
- `questions`：問題列表，每個問題包含 id、paragraph_id、question_text、answer_text、answer_start、answer_end
- `paragraphs`：段落列表（測試集的 answer 欄位為 null）

In [17]:
!unzip -o /content/hw7_train.json.zip -d /content/
!unzip -o /content/hw7_dev.json.zip -d /content/
!unzip -o /content/hw7_test.json.zip -d /content/


Archive:  /content/hw7_train.json.zip
  inflating: /content/hw7_train.json  
Archive:  /content/hw7_dev.json.zip
  inflating: /content/hw7_dev.json   
Archive:  /content/hw7_test.json.zip
  inflating: /content/hw7_test.json  


In [18]:
def read_data(file):
    with open(file, 'r', encoding="utf-8") as reader:
        data = json.load(reader)
    return data["questions"], data["paragraphs"]

train_questions, train_paragraphs = read_data("hw7_train.json")
dev_questions, dev_paragraphs = read_data("hw7_dev.json")
test_questions, test_paragraphs = read_data("hw7_test.json")

print(f"訓練集問題數：{len(train_questions)}")
print(f"驗證集問題數：{len(dev_questions)}")
print(f"測試集問題數：{len(test_questions)}")

訓練集問題數：26936
驗證集問題數：3524
測試集問題數：3493


## 7. Tokenize 資料

先分別 tokenize 問題和段落（不加 special tokens），
之後在 Dataset 的 `__getitem__` 再組合並加上 [CLS]、[SEP]。

In [19]:
# add_special_tokens=False：不自動加 [CLS]/[SEP]，之後手動控制
train_questions_tokenized = tokenizer(
    [q["question_text"] for q in train_questions], add_special_tokens=False
)
dev_questions_tokenized = tokenizer(
    [q["question_text"] for q in dev_questions], add_special_tokens=False
)
test_questions_tokenized = tokenizer(
    [q["question_text"] for q in test_questions], add_special_tokens=False
)

train_paragraphs_tokenized = tokenizer(train_paragraphs, add_special_tokens=False)
dev_paragraphs_tokenized = tokenizer(dev_paragraphs, add_special_tokens=False)
test_paragraphs_tokenized = tokenizer(test_paragraphs, add_special_tokens=False)

print("Tokenize 完成")
# 警告訊息可以忽略（序列超過 512 會警告，但我們在 Dataset 中會切 window）

Tokenize 完成


## 8. Dataset 與 DataLoader

**關鍵概念：Sliding Window**
- BERT 最大輸入長度 512（Self-Attention 是 O(n²)，太長會爆記憶體）
- 很多段落超過 512，需要用 sliding window 把段落切成多個小片段
- **訓練**：只取包含答案的那個 window
- **測試**：切成多個 window 各自預測，取信心分數最高的答案

**Medium 改動**：`doc_stride` 150 → 75，讓相鄰 window 有重疊
- 若答案在兩個 window 的邊界，重疊可確保至少有一個 window 完整包含答案

**Strong 改動**：Preprocessing 修正 `char_to_token` 可能回傳 `None` 的問題

In [20]:
class QA_Dataset(Dataset):
    def __init__(self, split, questions, tokenized_questions, tokenized_paragraphs):
        self.split = split
        self.questions = questions
        self.tokenized_questions = tokenized_questions
        self.tokenized_paragraphs = tokenized_paragraphs
        self.max_question_len = 40
        # Strong: 150 → 256，給模型更多段落上下文
        # 總長度 = 1 + 40 + 1 + 256 + 1 = 299，在 BERT 512 限制內
        self.max_paragraph_len = 256
        # doc_stride 設為 paragraph len 的一半，確保相鄰 window 重疊
        self.doc_stride = 128
        self.max_seq_len = 1 + self.max_question_len + 1 + self.max_paragraph_len + 1

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):
        question = self.questions[idx]
        tokenized_question = self.tokenized_questions[idx]
        tokenized_paragraph = self.tokenized_paragraphs[question["paragraph_id"]]

        if self.split == "train":
            answer_start_token = tokenized_paragraph.char_to_token(question["answer_start"])
            answer_end_token = tokenized_paragraph.char_to_token(question["answer_end"])

            # char_to_token 偶爾回傳 None，往前找最近的有效 token position
            if answer_end_token is None:
                for offset in range(1, 10):
                    answer_end_token = tokenized_paragraph.char_to_token(
                        question["answer_end"] - offset
                    )
                    if answer_end_token is not None:
                        break

            if answer_start_token is None or answer_end_token is None:
                answer_start_token = 0
                answer_end_token = 0

            mid = (answer_start_token + answer_end_token) // 2
            paragraph_start = max(
                0,
                min(mid - self.max_paragraph_len // 2,
                    len(tokenized_paragraph) - self.max_paragraph_len)
            )
            paragraph_end = paragraph_start + self.max_paragraph_len

            input_ids_question = [101] + tokenized_question.ids[:self.max_question_len] + [102]
            input_ids_paragraph = tokenized_paragraph.ids[paragraph_start:paragraph_end] + [102]

            answer_start_token += len(input_ids_question) - paragraph_start
            answer_end_token += len(input_ids_question) - paragraph_start

            input_ids, token_type_ids, attention_mask = self.padding(
                input_ids_question, input_ids_paragraph
            )
            return (
                torch.tensor(input_ids),
                torch.tensor(token_type_ids),
                torch.tensor(attention_mask),
                answer_start_token,
                answer_end_token,
            )

        else:
            input_ids_list, token_type_ids_list, attention_mask_list = [], [], []

            for i in range(0, len(tokenized_paragraph), self.doc_stride):
                input_ids_question = [101] + tokenized_question.ids[:self.max_question_len] + [102]
                input_ids_paragraph = tokenized_paragraph.ids[i:i + self.max_paragraph_len] + [102]
                input_ids, token_type_ids, attention_mask = self.padding(
                    input_ids_question, input_ids_paragraph
                )
                input_ids_list.append(input_ids)
                token_type_ids_list.append(token_type_ids)
                attention_mask_list.append(attention_mask)

            return (
                torch.tensor(input_ids_list),
                torch.tensor(token_type_ids_list),
                torch.tensor(attention_mask_list),
            )

    def padding(self, input_ids_question, input_ids_paragraph):
        padding_len = self.max_seq_len - len(input_ids_question) - len(input_ids_paragraph)
        input_ids = input_ids_question + input_ids_paragraph + [0] * padding_len
        token_type_ids = (
            [0] * len(input_ids_question)
            + [1] * len(input_ids_paragraph)
            + [0] * padding_len
        )
        attention_mask = (
            [1] * (len(input_ids_question) + len(input_ids_paragraph))
            + [0] * padding_len
        )
        return input_ids, token_type_ids, attention_mask


train_set = QA_Dataset("train", train_questions, train_questions_tokenized, train_paragraphs_tokenized)
dev_set = QA_Dataset("dev", dev_questions, dev_questions_tokenized, dev_paragraphs_tokenized)
test_set = QA_Dataset("test", test_questions, test_questions_tokenized, test_paragraphs_tokenized)

train_batch_size = 16

train_loader = DataLoader(train_set, batch_size=train_batch_size, shuffle=True, pin_memory=True)
dev_loader = DataLoader(dev_set, batch_size=1, shuffle=False, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=1, shuffle=False, pin_memory=True)

print(f"訓練集 batches：{len(train_loader)}")

訓練集 batches：1684


## 9. Evaluate 函式

**Boss Postprocessing 修正**：

原版 bug：
1. `start_index` 可能 > `end_index`（在 logits 各自取 argmax，沒有考慮相對順序）
2. 預測可能落在問題部分，而非段落部分

修正方式：
1. 找到 [SEP] 位置，只在段落範圍內預測
2. 確定 start 位置後，end 只考慮 >= start 的位置

In [21]:
def evaluate(data, output):
    answer = ''
    max_prob = float('-inf')
    num_of_windows = data[0].shape[1]

    for k in range(num_of_windows):
        input_ids = data[0][0][k]  # shape: (max_seq_len,)

        # Boss: 找到段落的起始/結束位置
        # 輸入格式：[CLS] question [SEP] paragraph [SEP]
        # 第一個 [SEP]（token id=102）之後才是段落
        sep_positions = (input_ids == 102).nonzero(as_tuple=False).squeeze(dim=1)
        if sep_positions.numel() >= 2:
            paragraph_start_pos = sep_positions[0].item() + 1  # 第一個 [SEP] 後
            paragraph_end_pos = sep_positions[1].item()         # 第二個 [SEP] 前
        else:
            # 防禦性處理：若找不到兩個 [SEP]，使用全序列
            paragraph_start_pos = 1
            paragraph_end_pos = len(input_ids) - 1

        start_logits = output.start_logits[k].clone()
        end_logits = output.end_logits[k].clone()

        # Boss: 將段落以外的位置設為 -inf，避免預測到問題或 padding
        start_logits[:paragraph_start_pos] = float('-inf')
        end_logits[:paragraph_start_pos] = float('-inf')
        start_logits[paragraph_end_pos:] = float('-inf')
        end_logits[paragraph_end_pos:] = float('-inf')

        start_prob, start_index = torch.max(start_logits, dim=0)

        # Boss: 確保 end >= start，避免倒序答案
        end_logits[:start_index] = float('-inf')
        end_prob, end_index = torch.max(end_logits, dim=0)

        prob = start_prob + end_prob

        if prob > max_prob:
            max_prob = prob
            # 將 token ids 轉回文字（中文 BERT tokenizer 會加空格，需去除）
            answer = tokenizer.decode(data[0][0][k][start_index:end_index + 1])

    return answer.replace(' ', '')

## 10. 訓練

**Medium 改動**：Linear LR decay with warmup
- Warmup：前 10% 的 steps 讓 LR 從 0 線性上升到設定值
- Decay：之後讓 LR 從設定值線性下降到 0
- 好處：避免訓練初期梯度爆炸，訓練後期精細調整參數

In [26]:
num_epoch = 3
validation = True
logging_step = 100
learning_rate = 2e-5   # BERT fine-tune 標準值

optimizer = AdamW(model.parameters(), lr=learning_rate)

total_steps = num_epoch * len(train_loader)
warmup_steps = int(0.1 * total_steps)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f"總訓練步數：{total_steps}，Warmup 步數：{warmup_steps}")

if fp16_training:
    model, optimizer, train_loader = accelerator.prepare(model, optimizer, train_loader)

model.train()
print("開始訓練...")

for epoch in range(num_epoch):
    step = 1
    train_loss = train_acc = 0

    for data in tqdm(train_loader, desc=f"Epoch {epoch + 1}"):
        data = [i.to(device) for i in data]

        output = model(
            input_ids=data[0],
            token_type_ids=data[1],
            attention_mask=data[2],
            start_positions=data[3],
            end_positions=data[4],
        )

        start_index = torch.argmax(output.start_logits, dim=1)
        end_index = torch.argmax(output.end_logits, dim=1)
        train_acc += ((start_index == data[3]) & (end_index == data[4])).float().mean()
        train_loss += output.loss

        if fp16_training:
            accelerator.backward(output.loss)
        else:
            output.loss.backward()

        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        step += 1

        if step % logging_step == 0:
            print(
                f"Epoch {epoch + 1} | Step {step} "
                f"| loss = {train_loss.item() / logging_step:.3f} "
                f"| acc = {train_acc / logging_step:.3f} "
                f"| lr = {scheduler.get_last_lr()[0]:.2e}"
            )
            train_loss = train_acc = 0

    if validation:
        print("評估 Dev Set...")
        model.eval()
        with torch.no_grad():
            dev_acc = 0
            for i, data in enumerate(tqdm(dev_loader, desc="Dev")):
                output = model(
                    input_ids=data[0].squeeze(dim=0).to(device),
                    token_type_ids=data[1].squeeze(dim=0).to(device),
                    attention_mask=data[2].squeeze(dim=0).to(device),
                )
                dev_acc += evaluate(data, output) == dev_questions[i]["answer_text"]
            print(f"Validation | Epoch {epoch + 1} | acc = {dev_acc / len(dev_loader):.3f}")
        model.train()

print("儲存模型...")
model_save_dir = "saved_model"
model.save_pretrained(model_save_dir)
print(f"模型已儲存至 {model_save_dir}/")

總訓練步數：8420，Warmup 步數：842
開始訓練...


Epoch 1:   0%|          | 0/1684 [00:00<?, ?it/s]

Epoch 1 | Step 100 | loss = 5.529 | acc = 0.001 | lr = 2.35e-06
Epoch 1 | Step 200 | loss = 4.626 | acc = 0.021 | lr = 4.73e-06
Epoch 1 | Step 300 | loss = 2.345 | acc = 0.300 | lr = 7.10e-06
Epoch 1 | Step 400 | loss = 1.254 | acc = 0.520 | lr = 9.48e-06
Epoch 1 | Step 500 | loss = 1.027 | acc = 0.605 | lr = 1.19e-05
Epoch 1 | Step 600 | loss = 0.842 | acc = 0.662 | lr = 1.42e-05
Epoch 1 | Step 700 | loss = 0.852 | acc = 0.661 | lr = 1.66e-05
Epoch 1 | Step 800 | loss = 0.788 | acc = 0.691 | lr = 1.90e-05
Epoch 1 | Step 900 | loss = 0.688 | acc = 0.718 | lr = 1.98e-05
Epoch 1 | Step 1000 | loss = 0.664 | acc = 0.719 | lr = 1.96e-05
Epoch 1 | Step 1100 | loss = 0.689 | acc = 0.721 | lr = 1.93e-05
Epoch 1 | Step 1200 | loss = 0.625 | acc = 0.754 | lr = 1.91e-05
Epoch 1 | Step 1300 | loss = 0.615 | acc = 0.744 | lr = 1.88e-05
Epoch 1 | Step 1400 | loss = 0.531 | acc = 0.769 | lr = 1.85e-05
Epoch 1 | Step 1500 | loss = 0.637 | acc = 0.740 | lr = 1.83e-05
Epoch 1 | Step 1600 | loss = 0.602

Dev:   0%|          | 0/3524 [00:00<?, ?it/s]

Validation | Epoch 1 | acc = 0.781


...

Epoch 5 | Step 100 | loss = 0.065 | acc = 0.949 | lr = 4.18e-06
Epoch 5 | Step 200 | loss = 0.070 | acc = 0.957 | lr = 3.92e-06
Epoch 5 | Step 300 | loss = 0.067 | acc = 0.961 | lr = 3.66e-06
Epoch 5 | Step 400 | loss = 0.081 | acc = 0.952 | lr = 3.39e-06
Epoch 5 | Step 500 | loss = 0.079 | acc = 0.952 | lr = 3.13e-06
Epoch 5 | Step 600 | loss = 0.075 | acc = 0.962 | lr = 2.86e-06
Epoch 5 | Step 700 | loss = 0.079 | acc = 0.955 | lr = 2.60e-06
Epoch 5 | Step 800 | loss = 0.092 | acc = 0.946 | lr = 2.34e-06
Epoch 5 | Step 900 | loss = 0.075 | acc = 0.962 | lr = 2.07e-06
Epoch 5 | Step 1000 | loss = 0.072 | acc = 0.959 | lr = 1.81e-06
Epoch 5 | Step 1100 | loss = 0.067 | acc = 0.957 | lr = 1.54e-06
Epoch 5 | Step 1200 | loss = 0.067 | acc = 0.961 | lr = 1.28e-06
Epoch 5 | Step 1300 | loss = 0.082 | acc = 0.943 | lr = 1.02e-06
Epoch 5 | Step 1400 | loss = 0.085 | acc = 0.944 | lr = 7.52e-07
Epoch 5 | Step 1500 | loss = 0.081 | acc = 0.956 | lr = 4.88e-07
Epoch 5 | Step 1600 | loss = 0.075

Dev:   0%|          | 0/3524 [00:00<?, ?it/s]

Validation | Epoch 5 | acc = 0.793
儲存模型...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

模型已儲存至 saved_model/


## 11. 測試並輸出結果

In [27]:
print("評估 Test Set...")
result = []

model.eval()
with torch.no_grad():
    for data in tqdm(test_loader, desc="Test"):
        output = model(
            input_ids=data[0].squeeze(dim=0).to(device),
            token_type_ids=data[1].squeeze(dim=0).to(device),
            attention_mask=data[2].squeeze(dim=0).to(device),
        )
        result.append(evaluate(data, output))

# 輸出 CSV（Kaggle 提交格式）
result_file = "result.csv"
with open(result_file, 'w') as f:
    f.write("ID,Answer\n")
    for i, test_question in enumerate(test_questions):
        # 答案中的逗號需去掉（因為 CSV 用逗號分隔）
        f.write(f"{test_question['id']},{result[i].replace(',', '')}\n")

print(f"完成！結果已寫入 {result_file}")

評估 Test Set...


Test:   0%|          | 0/3493 [00:00<?, ?it/s]

完成！結果已寫入 result.csv
